# 🤖 Modeling & Evaluation

**Goal:** Train two embedding variants, evaluate them, run A/B test, log with MLflow.

- **Variant A:** `all-MiniLM-L6-v2` — 384-dim, fast
- **Variant B:** `all-mpnet-base-v2` — 768-dim, more accurate

In [24]:
import sys
import importlib
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import os
import json
import numpy as np
import pandas as pd

# Force-reload evaluator so any source changes are picked up in this session
import course_recommender.models.evaluator as _ev_mod
importlib.reload(_ev_mod)

from course_recommender.models.recommender import ContentBasedRecommender
from course_recommender.models.evaluator import RecommenderEvaluator
from course_recommender.mlops.mlflow_utils import setup_mlflow, log_model_training
from course_recommender.mlops.ab_testing import ABTest
from course_recommender.utils.config import config

os.makedirs('../models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
print('✅ Imports OK')

✅ Imports OK


## 1. Load Cleaned Data

In [25]:
courses_df = pd.read_csv('../data/processed/courses_clean.csv')
print(f'Loaded {len(courses_df)} courses')
print(f'Columns: {list(courses_df.columns)}')

Loaded 6089 courses
Columns: ['Course Title', 'Rating', 'Level', 'Schedule', 'What you will learn', 'Skill gain', 'Modules', 'Instructor', 'Offered By', 'Keyword', 'Course Url', 'Duration to complete (Approx.)', 'Number of Review', 'course_id', 'combined_text', 'has_rating', 'num_skills', 'text_length']


In [26]:
# Dataset overview + category distribution
print(f'Shape: {courses_df.shape}')
print(f'\nUnique difficulty levels ({courses_df["Level"].nunique()}): {courses_df["Level"].unique().tolist()}')
print(f'\nUnique subject categories ({courses_df["Keyword"].nunique()}):')
print(courses_df['Keyword'].value_counts().to_string())
print(f'\nCourses with "Not specified" description: {(courses_df["What you will learn"] == "Not specified").sum()}')
print(f'Avg skills per course: {courses_df["num_skills"].mean():.1f}')

Shape: (6089, 18)

Unique difficulty levels (4): ['Beginner', 'Intermediate', 'Not specified', 'Advanced']

Unique subject categories (10):
Keyword
Business                            930
Computer Science                    913
Health                              797
Information Technology              709
Personal Development                641
Physical Science and Engineering    608
Social Sciences                     529
Arts and Humanities                 476
DataScience                         318
Math and Logic                      168

Courses with "Not specified" description: 3050
Avg skills per course: 3.4


In [27]:
# Verify combined_text is correctly built (title + description + skills + level)
print('Sample combined_text entries:')
for _, row in courses_df.head(3).iterrows():
    print(f'\n[{row["course_id"]}] {row["Course Title"]}')
    print(f'  → {row["combined_text"][:120]}...')

Sample combined_text entries:

[0] Fashion as Design
  → Fashion as Design Not specified Art History, Art, History, Creativity Beginner...

[1] Modern American Poetry
  → Modern American Poetry Not specified Not specified Beginner...

[2] Pixel Art for Video Games
  → Pixel Art for Video Games Not specified Not specified Beginner...


## 2. Setup MLflow

In [28]:
# Use mlruns in project root for notebooks
import mlflow
mlflow.set_tracking_uri('sqlite:///../mlflow.db')
mlflow.set_experiment('course-recommender-ab-test')
print('✅ MLflow configured')

✅ MLflow configured


## 3. Train Variant A — all-MiniLM-L6-v2

In [29]:
print('Training Variant A...')
rec_a = ContentBasedRecommender(model_name='all-MiniLM-L6-v2')
rec_a.fit(courses_df, text_column='combined_text')

# Save embeddings
np.save('../data/processed/embeddings_variant_a.npy', rec_a.get_embeddings())
print(f'Embeddings shape: {rec_a.get_embeddings().shape}')

Training Variant A...
2026-04-18 20:09:39 | INFO     | course_recommender.models.recommender | Loading sentence-transformer model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-04-18 20:09:43 | INFO     | course_recommender.models.recommender | Encoding 6089 courses...


Batches:   0%|          | 0/96 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Evaluate Variant A
eval_a = RecommenderEvaluator(rec_a)
metrics_a = eval_a.evaluate(sample_size=100, top_k=5)
print('Variant A metrics:', metrics_a)

2026-04-18 19:27:23 | INFO     | course_recommender.models.evaluator | Evaluating recommender (all-MiniLM-L6-v2) — sample_size=100, top_k=5
2026-04-18 19:27:26 | INFO     | course_recommender.models.evaluator | Metrics: {'diversity': 0.008, 'coverage': 0.0793, 'avg_similarity': 0.6394, 'inference_time_ms': 9.38}
Variant A metrics: {'diversity': 0.008, 'coverage': 0.0793, 'avg_similarity': 0.6394, 'inference_time_ms': 9.38}


In [ ]:
# Save model
model_path_a = '../models/recommender_variant_a.pkl'
rec_a.save(model_path_a)

# Log to MLflow
run_id_a = log_model_training('A', 'all-MiniLM-L6-v2', rec_a, metrics_a, model_path_a)
print(f'MLflow Run ID (A): {run_id_a}')

2026-04-18 19:28:20 | INFO     | course_recommender.models.recommender | Saved recommender to '../models/recommender_variant_a.pkl'
2026-04-18 19:28:20 | INFO     | course_recommender.mlops.mlflow_utils | MLflow run logged — Variant A | Run ID: 144ae8f778f94bedb950527748a264ed
MLflow Run ID (A): 144ae8f778f94bedb950527748a264ed


## 4. Train Variant B — all-mpnet-base-v2

In [ ]:
print('Training Variant B (this is slower)...')
rec_b = ContentBasedRecommender(model_name='all-mpnet-base-v2')
rec_b.fit(courses_df, text_column='combined_text')

np.save('../data/processed/embeddings_variant_b.npy', rec_b.get_embeddings())
print(f'Embeddings shape: {rec_b.get_embeddings().shape}')

Training Variant B (this is slower)...
2026-04-18 19:28:22 | INFO     | course_recommender.models.recommender | Loading sentence-transformer model: all-mpnet-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-04-18 19:28:25 | INFO     | course_recommender.models.recommender | Encoding 6089 courses...


Batches:   0%|          | 0/96 [00:00<?, ?it/s]

2026-04-18 19:33:23 | INFO     | course_recommender.models.recommender | Embeddings shape: (6089, 768)  |  fit time: 297.6s
Embeddings shape: (6089, 768)


In [ ]:
eval_b = RecommenderEvaluator(rec_b)
metrics_b = eval_b.evaluate(sample_size=100, top_k=5)
print('Variant B metrics:', metrics_b)

2026-04-18 19:33:23 | INFO     | course_recommender.models.evaluator | Evaluating recommender (all-mpnet-base-v2) — sample_size=100, top_k=5
2026-04-18 19:33:29 | INFO     | course_recommender.models.evaluator | Metrics: {'diversity': 0.008, 'coverage': 0.078, 'avg_similarity': 0.691, 'inference_time_ms': 21.02}
Variant B metrics: {'diversity': 0.008, 'coverage': 0.078, 'avg_similarity': 0.691, 'inference_time_ms': 21.02}


In [ ]:
model_path_b = '../models/recommender_variant_b.pkl'
rec_b.save(model_path_b)

run_id_b = log_model_training('B', 'all-mpnet-base-v2', rec_b, metrics_b, model_path_b)
print(f'MLflow Run ID (B): {run_id_b}')

2026-04-18 19:33:29 | INFO     | course_recommender.models.recommender | Saved recommender to '../models/recommender_variant_b.pkl'
2026-04-18 19:33:29 | INFO     | course_recommender.mlops.mlflow_utils | MLflow run logged — Variant B | Run ID: fac13fa5e5ff44ccabc97927142d570f
MLflow Run ID (B): fac13fa5e5ff44ccabc97927142d570f


## 5. A/B Test Comparison

In [ ]:
ab = ABTest(rec_a, rec_b)
comparison = ab.compare_metrics(sample_size=100, top_k=5)
print(comparison.drop(columns=['key']).to_string(index=False))

print("""
Metric notes:
  Diversity    — fraction of unique subject categories across all recommendation lists.
                 Low by design: content-based systems cluster tightly within topics.
  Coverage     — fraction of the 6089-course catalog ever recommended (100-course sample).
  Avg Similarity — mean cosine similarity; higher = tighter semantic match.
  Inference Time — ms per recommendation request (CPU).
""")

In [ ]:
fig = ab.visualize_comparison()
fig.show()

In [ ]:
winner = ab.determine_winner(primary_metric='avg_similarity')
print(f'\n🏆 Winner by avg_similarity: Variant {winner}')

2026-04-18 19:34:57 | INFO     | course_recommender.mlops.ab_testing | Winner (by avg_similarity): Variant B  [A=0.6356, B=0.6876]

🏆 Winner by avg_similarity: Variant B


## 6. Qualitative Analysis

In [ ]:
# Sample recommendations for a Python course
name_col = next((c for c in courses_df.columns if 'name' in c.lower() or 'title' in c.lower()), None)
if name_col:
    python_courses = courses_df[courses_df[name_col].str.contains('Python', na=False, case=False)]
    if not python_courses.empty:
        seed_id = int(python_courses.iloc[0]['course_id'])
        seed_name = python_courses.iloc[0][name_col]
        print(f'Seed course: [{seed_id}] {seed_name}')
        print('\nTop 5 similar (Variant A):')
        recs_a = rec_a.recommend_similar(seed_id, top_k=5)
        for _, r in recs_a.iterrows():
            print(f'  {r[name_col][:60]:60s}  sim={r["similarity_score"]:.3f}')

Seed course: [464] Data Processing Using Python

Top 5 similar (Variant A):
  Python for Data Analysis: Pandas & NumPy                      sim=0.682
  Data Science with NumPy, Sets, and Dictionaries               sim=0.641
  Data Analysis with Python                                     sim=0.615
  Data Analysis Using Python                                    sim=0.608
  Expressway to Data Science: Python Programming Specializatio  sim=0.603


In [ ]:
# Search test
print('Search: "deep learning computer vision"')
results = rec_a.search('deep learning computer vision', top_k=5)
if name_col and name_col in results.columns:
    for _, r in results.iterrows():
        print(f'  {r[name_col][:60]:60s}  sim={r["similarity_score"]:.3f}')

Search: "deep learning computer vision"
  Computer Vision for Engineering and Science Specialization    sim=0.614
  Neuronale Netze und Deep Learning                             sim=0.542
  Object Tracking and Motion Detection with Computer Vision     sim=0.538
  DeepLearning.AI TensorFlow Developer Professional Certificat  sim=0.520
  Building Deep Learning Models with TensorFlow                 sim=0.511


## 7. Save Metadata

In [ ]:
metadata = {
    'variant_a': {
        'model_name': rec_a.model_name,
        'num_courses': rec_a.num_courses,
        'embedding_dim': rec_a.embedding_dim,
        'metrics': metrics_a,
        'mlflow_run_id': run_id_a,
    },
    'variant_b': {
        'model_name': rec_b.model_name,
        'num_courses': rec_b.num_courses,
        'embedding_dim': rec_b.embedding_dim,
        'metrics': metrics_b,
        'mlflow_run_id': run_id_b,
    },
    'winner': winner,
}

with open('../models/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✅ Saved models/metadata.json')
print(json.dumps(metadata, indent=2))